# MiniGPT-Stack Phase 6 / S4 - layer-4 SAE GPU PROBE (not the acceptance run)

Pre-registered in `docs/phase6_s4_probe.md` (stop rule, measurements, estimate formula). Rules for
the real run: `docs/phase6_preregistration.md` (incl. Amendment 1: window position 0 is excluded
from SAE training). Everything below is labeled **probe, not acceptance**.

- Hard cap **30 minutes** (`faulthandler` watchdog kills the kernel at the cap; stages also check the
  time left). Training budget 6 minutes, stopped at an epoch boundary.
- SHA-256 gate on the base checkpoint. Only the fast tests run. If FVU is not falling clearly the
  report says STOP and nothing is scaled up.

**Attach two datasets:** `glassbox-sft-init-weights` (contains `weights_step12158.pt`, the base
model) and `tinystories-shards-8192`.


In [ ]:
# ---- PINNED (see docs/phase6_s4_probe.md) ----
CONFIG        = "configs/base_30m.yaml"
DEVICE        = "cuda"
LAYER         = 4
N_FEATURES    = 4096
K             = 32
LR            = 4e-4
WARMUP        = 50
BATCH_TOKENS  = 4096
BUFFER_TOKENS = 524_288
EPOCHS        = 24            # planned; stopped early by the training budget
CTX           = 512
SEED          = 0
NORM_WINDOWS  = 256
EVAL_WINDOWS  = 64            # per-epoch quick eval (held-out validation windows)
EVAL_TOKENS   = 500_000       # A1-A4 evaluation tokens
DEAD_TOKENS   = 10_000_000    # A3 scan (reduced only if the time cap requires it; flagged)
TRAIN_BUDGET_H = 0.1          # 6 minutes
FVU_RATIO, FVU_DEC_FRAC, MIN_EPOCHS = 0.85, 0.70, 4      # the pre-registered stop rule
SMOKE_SCALE   = 0.3446        # scale from the 1,000-story smoke shard, for comparison

WEIGHTS_FILE   = "weights_step12158.pt"
WEIGHTS_BYTES  = 125_893_589
WEIGHTS_SHA256 = "feb065a5bc7d0ca240d756cd7bc48d8e1e284349a35dcd783ece4dc0c899aab0"

import faulthandler, time
T0 = time.time()
HARD_CAP_S = 30 * 60
faulthandler.dump_traceback_later(HARD_CAP_S, exit=True)     # hard cap: the process exits at 30 min
COMPLETED = False


def time_left():
    return HARD_CAP_S - (time.time() - T0)


def capped(argv):
    import subprocess
    left = time_left()
    if left < 30:
        raise RuntimeError(f"[probe] hard cap: only {left:.0f}s left; not starting {argv[:4]}")
    return subprocess.run(argv, timeout=left)


## GPU check

In [ ]:
import torch
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU (T4) in notebook settings"


## Clone

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

## Install

In [ ]:
# Kaggle already ships numpy, pyyaml, tqdm, pytest, regex and datasets.
!pip install -q regex pyyaml pytest tqdm datasets


## Inputs - base checkpoint, byte-exact and SHA-256 gate

In [ ]:
import glob, hashlib, os, torch

hits = glob.glob("/kaggle/input/**/" + WEIGHTS_FILE, recursive=True)
assert hits, "attach the private dataset glassbox-sft-init-weights (contains weights_step12158.pt)"
W = hits[0]
got = os.path.getsize(W)
print(W, "| bytes", got, "(expected", WEIGHTS_BYTES, ")")
assert got == WEIGHTS_BYTES, "weights size mismatch (truncated upload?)"
h = hashlib.sha256()
with open(W, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("[probe] sha256 expected  ", WEIGHTS_SHA256)
print("[probe] sha256 in-kernel ", h.hexdigest(), "->", "MATCH" if h.hexdigest() == WEIGHTS_SHA256 else "MISMATCH")
assert h.hexdigest() == WEIGHTS_SHA256, "SHA-256 mismatch; stopping before anything runs"
SHA_OK = True

from common.config import load_config
from model.gpt import GPT, count_params
cfg = load_config(CONFIG)
model = GPT(cfg["model"])
model.load_state_dict(torch.load(W, map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()
print("base weights load into the configured arch | params:", count_params(model), "| device", DEVICE)


## Shards (real train + validation)

In [ ]:
import glob, json, os
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, "attach the shards dataset first"
print("meta.json candidates:", hits)
shard_dir = os.path.dirname(hits[0])
print("--- meta.json head ---")
print("\n".join(open(os.path.join(shard_dir, "meta.json")).read().splitlines()[:15]))
os.makedirs("data", exist_ok=True)
# The cloned repo can contain a stale data/meta.json (smoke-test counts). If it is left in
# place, the symlink below is skipped and training would read the wrong token counts.
for f in glob.glob("data/*.bin") + ["data/meta.json"]:
    if os.path.lexists(f):
        os.remove(f)
for f in glob.glob(os.path.join(shard_dir, "*.bin")) + [os.path.join(shard_dir, "meta.json")]:
    os.symlink(f, os.path.join("data", os.path.basename(f)))
print("linked:", sorted(os.listdir("data")))
meta = json.load(open("data/meta.json"))
assert "shards" in meta and "vocab_size" in meta, meta.keys()
for s in meta["shards"]:
    assert {"split", "index", "file", "tokens"} <= set(s), s
print("meta keys OK:", sorted(meta.keys()))
tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
for s in meta["shards"]:
    nbytes = os.path.getsize(os.path.join("data", s["file"]))
    print(f"  {s['file']} {s['tokens']} toks {nbytes/1024/1024:.1f}MB")
    assert nbytes == 2 * s["tokens"], f"{s['file']}: file size does not match meta (stale meta.json?)"
assert tr > 400_000_000, f"train tokens {tr} look wrong (expected ~467M)"
TRAIN_TOKS, VAL_TOKS = tr, va

## Fast tests - fatal on failure

In [ ]:
import subprocess, sys

proc = capped([sys.executable, "-m", "pytest", "tests/test_hooks.py", "tests/test_model.py",
               "tests/test_interp_activations.py", "tests/test_interp_sae.py", "tests/test_interp_train.py",
               "tests/test_interp_eval.py", "-q"])
if proc.returncode != 0:
    raise RuntimeError(f"[probe] fast tests FAILED (rc={proc.returncode}); stopping")
TESTS_OK = True
print("[probe] fast tests: PASS", f"| {time.time() - T0:.0f}s elapsed", flush=True)


## Probe 1 - normalization scale on real windows, norm stats, extraction speed

The scale is computed over window positions >= 1 (Amendment 1). Extraction speed is `fill_buffer`
(fp32) for several chunk sizes: 1 warm-up then the median of 3, GPU synchronized.

In [ ]:
import json, numpy as np
from interp import activations as A
from train.data_loader import ShardLoader

tr = ShardLoader(cfg["data"]["out_dir"], "train")
va = ShardLoader(cfg["data"]["out_dir"], "validation")
PROBE = {"layer": LAYER, "label": "probe, not acceptance"}

SCALE = A.estimate_norm_scale(model, tr, LAYER, SEED, NORM_WINDOWS, CTX, 64, DEVICE)
stats = A.norm_stats(model, tr, LAYER, SEED, NORM_WINDOWS, CTX, 64, DEVICE)
PROBE["scale"] = SCALE
PROBE["scale_smoke_shard"] = SMOKE_SCALE
PROBE["norm_stats"] = stats
print(f"[probe] scale (real windows, positions >= 1) = {SCALE:.5f}  (smoke shard: {SMOKE_SCALE})")
print("[probe] layer-4 norms:", {k: round(v, 2) for k, v in stats.items() if "norm" in k or "max" in k})


def timed_fill(chunk):
    torch.cuda.synchronize(); t = time.perf_counter()
    A.fill_buffer(model, tr, LAYER, SEED, 12345, BUFFER_TOKENS, CTX, SCALE, chunk, DEVICE)
    torch.cuda.synchronize()
    return time.perf_counter() - t


ext = {}
rows = A.buffer_rows(BUFFER_TOKENS, CTX)
for chunk in (32, 64, 128):
    timed_fill(chunk)                                           # warm-up
    ts = sorted(timed_fill(chunk) for _ in range(3))
    ext[chunk] = {"median_s": ts[1], "tokens_per_s": rows / ts[1]}
    print(f"[probe] extraction chunk={chunk:>3}: {ts[1]:.2f}s per {rows:,} rows -> {rows / ts[1]:,.0f} tok/s")
PROBE["extraction"] = ext
BEST_CHUNK = max(ext, key=lambda c: ext[c]["tokens_per_s"])
print("[probe] best extraction chunk:", BEST_CHUNK, f"| {time.time() - T0:.0f}s elapsed", flush=True)


## Probe 2 - short SAE training run (layer 4, TopK, position 0 excluded)

Stops at an epoch boundary once the training budget is used. FVU and L0 on held-out validation
windows after every epoch; per-epoch fill/train seconds, tokens/s and peak GPU memory.

In [ ]:
from interp.train_sae import TrainConfig, train

budget_h = max(0.02, min(TRAIN_BUDGET_H, (time_left() - 12 * 60) / 3600))
tcfg = TrainConfig(layer=LAYER, n_features=N_FEATURES, k=K, seed=SEED, lr=LR, warmup_steps=WARMUP,
                   batch_tokens=BATCH_TOKENS, buffer_tokens=BUFFER_TOKENS, epochs=EPOCHS, ctx=CTX,
                   norm_windows=NORM_WINDOWS, chunk=BEST_CHUNK, log_every=25, eval_every=1,
                   ckpt_every=4, eval_windows=EVAL_WINDOWS, max_hours=budget_h)
OUT = "/kaggle/working/sae_probe"
print(f"[probe] training budget {budget_h * 60:.1f} min, planned {EPOCHS} epochs of {BUFFER_TOKENS:,} tokens", flush=True)
t_tr = time.time()
res = train(model, tr, va, tcfg, OUT, device=DEVICE, log=print,
            meta_extra={"label": "probe, not acceptance", "weights_sha256": WEIGHTS_SHA256})
TRAIN_WALL = time.time() - t_tr
rows_m = [json.loads(l) for l in open(OUT + "/metrics_sae.jsonl", encoding="utf-8") if l.strip()]
evals = [(r["epoch"] + 1, r["eval"]) for r in rows_m if "eval" in r]
stats_e = [r["epoch_stats"] for r in rows_m if "epoch_stats" in r]
term = [r for r in rows_m if "terminal" in r][-1]
fvu = [e["fvu"] for _, e in evals]
n_pairs = max(len(fvu) - 1, 1)
dec = sum(1 for a, b in zip(fvu, fvu[1:]) if b < a)
falling = (len(fvu) >= MIN_EPOCHS and fvu[-1] <= FVU_RATIO * fvu[0] and dec / n_pairs >= FVU_DEC_FRAC)
med = lambda xs: float(np.median(xs))
PROBE["training"] = {
    "epochs_completed": len(fvu), "terminal": term["terminal"], "wall_s": TRAIN_WALL,
    "budget_min": budget_h * 60,
    "fill_s_median": med([s["fill_s"] for s in stats_e]), "train_s_median": med([s["train_s"] for s in stats_e]),
    "epoch_s_min": min(s["fill_s"] + s["train_s"] for s in stats_e),
    "epoch_s_max": max(s["fill_s"] + s["train_s"] for s in stats_e),
    "train_tokens_per_s_median": med([s["train_tokens_per_s"] for s in stats_e]),
    "extract_tokens_per_s_median": med([s["extract_tokens_per_s"] for s in stats_e]),
    "peak_mem_mb": max(s.get("peak_mem_mb", 0) for s in stats_e),
    "fvu_trajectory": [{"epoch": e, **ev} for e, ev in evals],
    "stop_rule": {"fvu_first": fvu[0], "fvu_last": fvu[-1], "ratio": fvu[-1] / fvu[0],
                  "decreasing_pairs": f"{dec}/{n_pairs}", "falling_clearly": bool(falling)}}
for e, ev in evals:
    print(f"[probe] epoch {e:>2}: fvu {ev['fvu']:.4f} (excl pos0 {ev.get('fvu_excl_pos0', float('nan')):.4f}) "
          f"l0 {ev['l0']:.1f} alive {ev['alive_frac']:.3f}")
print(f"[probe] train throughput {PROBE['training']['train_tokens_per_s_median']:,.0f} tok/s | extraction "
      f"{PROBE['training']['extract_tokens_per_s_median']:,.0f} tok/s | peak mem {PROBE['training']['peak_mem_mb']:.0f} MB")
print(f"[probe] stop rule: last/first = {fvu[-1] / fvu[0]:.3f} (need <= {FVU_RATIO}), decreasing pairs {dec}/{n_pairs} "
      f"(need >= {FVU_DEC_FRAC:.0%}), epochs {len(fvu)} (need >= {MIN_EPOCHS})  ->  "
      f"{'FALLING CLEARLY' if falling else 'FVU NOT FALLING CLEARLY: STOP'}", flush=True)


## Probe 3 - A1-A4 on the short-run SAE (labeled **probe, not acceptance**)

Real definitions: 500,000 evaluation tokens, the dead-feature scan (reduced from 10M only if the time
cap requires it, and flagged), CE recovered with the mean-ablation baseline, plus the position-0
extras of Amendment 1. A5 is not run (no SFT model attached).

In [ ]:
from interp import eval_sae as E
from interp.sae import load_sae
from tokenizer.bpe import BPETokenizer

sae, meta = load_sae(OUT + "/sae_latest.pt")
sae = sae.to(DEVICE).eval()
dead_tokens = DEAD_TOKENS if time_left() > 11 * 60 else 2_000_000
PROBE["dead_tokens_used"] = dead_tokens
PROBE["dead_scan_reduced"] = dead_tokens != DEAD_TOKENS
torch.cuda.synchronize(); t_ev = time.time()
rep = E.evaluate(model, sae, meta["act_scale"], LAYER, va, tr, CTX,
                 BPETokenizer.load(cfg["data"]["tokenizer_path"]), cfg["data"]["out_dir"],
                 eval_tokens=EVAL_TOKENS, dead_tokens=dead_tokens, sft_model=None, chunk=BEST_CHUNK)
torch.cuda.synchronize()
EVAL_S = time.time() - t_ev
rep["label"] = "probe, not acceptance"
rep["eval_seconds"] = EVAL_S
PROBE["a1_a4_probe"] = rep
print(f"[probe] (probe, not acceptance) A1 fvu {rep['fvu']:.4f} | A2 l0 {rep['l0']:.2f} | A3 dead {rep['dead_fraction']:.3f} "
      f"({rep['scan_tokens']:,} tokens) | A4 ce recovered {rep['ce_recovered']:.3f} | eval took {EVAL_S:.0f}s")
print("[probe] fvu by position:", {k: round(v, 4) for k, v in rep["fvu_by_position"].items()})
print("[probe] excl pos0:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rep["excl_pos0"].items()})
print("[probe] verdict flags (probe):", rep["verdict"], "| excl pos0:", rep["verdict_excl_pos0"], flush=True)


## Estimate for the full 100M-token run, report, COMPLETE guard

In [ ]:
import math

tr_ = PROBE["training"]
rows_per_epoch = A.buffer_rows(BUFFER_TOKENS, CTX)
epochs_needed = math.ceil(100_000_000 / rows_per_epoch)
epoch_med = tr_["fill_s_median"] + tr_["train_s_median"]
est = {"epochs_needed": epochs_needed, "rows_per_epoch": rows_per_epoch,
       "epoch_s_median": epoch_med,
       "train_hours_median": epochs_needed * epoch_med / 3600,
       "train_hours_range": [epochs_needed * tr_["epoch_s_min"] / 3600, epochs_needed * tr_["epoch_s_max"] / 3600],
       "full_eval_hours": EVAL_S / 3600, "setup_hours": 4 / 60}
est["total_hours_median"] = est["train_hours_median"] + est["full_eval_hours"] + est["setup_hours"]
PROBE["full_run_estimate"] = est
PROBE["kernel_elapsed_s"] = time.time() - T0
PROBE["hash_gate"] = "PASS" if SHA_OK else "FAIL"
PROBE["tests"] = "PASS" if TESTS_OK else "FAIL"
json.dump(PROBE, open("/kaggle/working/probe_report.json", "w"), indent=1, default=float)
lines = ["# Phase 6 S4 probe summary (probe, not acceptance)", "",
         f"- hash gate {PROBE['hash_gate']}, fast tests {PROBE['tests']}, kernel elapsed {PROBE['kernel_elapsed_s']:.0f}s of {HARD_CAP_S}s",
         f"- scale {SCALE:.5f} (smoke shard {SMOKE_SCALE}); extraction tok/s by chunk: "
         + ", ".join(f"{c}: {v['tokens_per_s']:,.0f}" for c, v in ext.items()),
         f"- training: {tr_['epochs_completed']} epochs, terminal {tr_['terminal']}, train {tr_['train_tokens_per_s_median']:,.0f} tok/s, "
         f"peak mem {tr_['peak_mem_mb']:.0f} MB",
         f"- stop rule: {tr_['stop_rule']}",
         f"- full-run estimate: {est['epochs_needed']} epochs x {est['epoch_s_median']:.1f}s = {est['train_hours_median']:.2f} h "
         f"(range {est['train_hours_range'][0]:.2f}-{est['train_hours_range'][1]:.2f}) + eval {est['full_eval_hours']:.2f} h + setup 0.07 h = "
         f"{est['total_hours_median']:.2f} GPU h"]
open("/kaggle/working/probe_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
COMPLETED = True
faulthandler.cancel_dump_traceback_later()


## COMPLETE guard - never claims success after a crash

In [ ]:
if COMPLETED:
    print("=" * 78)
    print("[probe] COMPLETE (probe, not acceptance)")
    print("=" * 78, flush=True)
else:
    raise RuntimeError("[probe] COMPLETED sentinel is False; NOT reporting COMPLETE.")
